# 02 Experiments (protocol v2): re-running the method described in the dissertation

Runs in Google Colab (GPU recommended). Implements dissertation Chapter 4 as the specification, with a clean protocol:

1. **Training pool** = DAIC-WOZ train + dev (140). **Held-out**: DAIC-WOZ official test (46) and E-DAIC (112), each scored once.
2. Architecture and graph hyperparameters fixed at the values the dissertation reports (Table 9). Optuna (TPE) tunes
   learning rate, weight decay, epochs and the class-weight multiplier m, by repeated stratified 5-fold CV inside the pool.
3. **Cross-fitted ensemble**: 5 folds x 4 seeds = 20 members. The threshold is calibrated on the out-of-fold probabilities
   of these same members (fewest false negatives with false-positive rate <= 6/21).

v1 (results in `results/experiments/`) tuned 9 hyperparameters on 107 participants and calibrated the threshold on
different models than the ones that scored the cohorts; its operating points were unstable. v2 results go to
`results/experiments/v2/`.

Deliberate changes from the legacy code: paralinguistic tags counted correctly, no 0.5 s turn filter, tags removed from the
lexical view, psycholinguistic view from `data/mapping/phq8_dsm5_empath.json`.
Only aggregates are written to the repository. Per-participant outputs go to `OUTPUT_DIR`, outside it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, sys
REPO = '/content/drive/MyDrive/05_Mestrado/Repositórios/depression-gnn-local/depression-gnn'
os.environ['DAIC_DATA_DIR'] = '/content/drive/MyDrive/Colab Notebooks/datasets'
OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/outputs/paper1/v2'   # outside the repository
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(REPO)
sys.path.insert(0, REPO)

In [ ]:
!pip install -q -r requirements.txt
!python -m pytest -q tests

.......................                                                  [100%]
=============================== warnings summary ===============================
tests/test_experiment.py::test_run_final[inductgcn-words-views0]
  /content/drive/MyDrive/05_Mestrado/Repositórios/depression-gnn-local/depression-gnn/src/paper1_graph_model/models.py:53: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
    a = torch.sparse_coo_tensor(edge_index, edge_weight, (x.shape[0], x.shape[0]))

tests/test_experiment.py::test_run_final[gcn_gat-words-views2]
tests/test_experiment.py::test_run_final[gcn_gat-words-views2]
tests/test_experiment.py::test_run_final[gat

## Configuration

In [ ]:
import torch
N_TRIALS = 30          # Optuna trials per model (restricted space: 4 hyperparameters)
N_FOLDS = 5
N_REPEATS = 2          # repeated CV inside the objective
SEEDS_PER_FOLD = 4     # ensemble = N_FOLDS * SEEDS_PER_FOLD = 20 members
SEED = 42
MAX_FPR = 6 / 21
SPACE = 'restricted'   # or 'full' for all of dissertation Table 9
STUDY_VERSION = 'v2'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
STORAGE = f'sqlite:///{OUTPUT_DIR}/optuna.db'   # resumable if the runtime disconnects
print(DEVICE, torch.__version__)

cuda 2.11.0+cu130


## Cohorts and features

In [ ]:
from src.common import data
from src.common.features import EmpathScorer
from src.paper1_graph_model.corpus import build_cohorts
from src.paper1_graph_model.experiment import merge_cohorts

scorer = EmpathScorer()
cohorts = build_cohorts(data.data_root(), scorer, psycholinguistic='dsm5')
pool = merge_cohorts('daic_pool', cohorts['daic_train'], cohorts['daic_dev'])
held_out = [cohorts['daic_test'], cohorts['edaic']]
for c in [pool] + held_out:
    print(f'{c.name:12s} n={len(c.y):4d} positives={int(c.y.sum()):3d}')

daic_pool    n= 140 positives= 42
daic_test    n=  46 positives= 14
edaic        n= 112 positives= 34


## Models (rows of dissertation Table 11)

In [ ]:
from src.paper1_graph_model.experiment import ModelSpec
ALL = ('temporal', 'paralinguistic', 'psycholinguistic')
CATS = scorer.categories
SPECS = [
    ModelSpec('InductGCN',                      'inductgcn', 'words',  ()),
    ModelSpec('GAT text-only',                  'gat',       'words',  ()),
    ModelSpec('GCN+GAT hybrid',                 'gcn_gat',   'words',  ALL),
    ModelSpec('EmpathGCN',                      'gcn',       'empath', ('psycholinguistic',), CATS),
    ModelSpec('GAT-GCN-FeExtr (Empath graph)',  'gat',       'empath', ALL, CATS),
    ModelSpec('GAT-GCN-FeExtr (word graph)',    'gat',       'words',  ALL),
]

## Hyperparameter search (training pool only)

In [ ]:
import json, pathlib
from src.paper1_graph_model.experiment import tune, hp_from_params
RESULTS = pathlib.Path('results/experiments/v2'); RESULTS.mkdir(parents=True, exist_ok=True)

best = {}
for spec in SPECS:
    study = tune(spec, pool, N_TRIALS, N_FOLDS, N_REPEATS, SEED, DEVICE, storage=STORAGE,
                 space=SPACE, study_name=f'{spec.name} [{STUDY_VERSION}]')
    best[spec.name] = {'params': study.best_params, 'cv_pr_auc': study.best_value,
                       'n_trials': len(study.trials), 'space': SPACE}
    print(f'{spec.name:32s} CV PR-AUC={study.best_value:.3f}  {study.best_params}')
(RESULTS / 'best_params.json').write_text(json.dumps(best, indent=2))

[I 2026-10-19 01:42:53,150] A new study created in RDB with name: InductGCN [v2]
/content/drive/MyDrive/05_Mestrado/Repositórios/depression-gnn-local/depression-gnn/src/paper1_graph_model/models.py:53: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  a = torch.sparse_coo_tensor(edge_index, edge_weight, (x.shape[0], x.shape[0]))
[I 2026-10-19 01:43:12,687] Trial 0 finished with value: 0.47483264567672756 and parameters: {'lr': 0.0005611516415334506, 'weight_decay': 0.0039847274093219665, 'epochs': 550, 'm': 3.79597545259111}. Best is trial 0 with value: 0.47483264567672756.
[I 2026-10-19 01:43:20,756] Trial 1 finished with value: 0.44982495272

InductGCN                        CV PR-AUC=0.519  {'lr': 0.0004059611610484307, 'weight_decay': 0.0005603803105930283, 'epochs': 450, 'm': 2.8736874205941256}


[I 2026-10-19 01:49:53,317] Trial 0 finished with value: 0.4476358557314786 and parameters: {'lr': 0.0005611516415334506, 'weight_decay': 0.0039847274093219665, 'epochs': 550, 'm': 3.79597545259111}. Best is trial 0 with value: 0.4476358557314786.
[I 2026-10-19 01:50:14,474] Trial 1 finished with value: 0.4741225222555683 and parameters: {'lr': 0.0002051338263087451, 'weight_decay': 0.00010255552094216992, 'epochs': 300, 'm': 4.598528437324806}. Best is trial 1 with value: 0.4741225222555683.
[I 2026-10-19 01:50:35,583] Trial 2 finished with value: 0.44097147334151643 and parameters: {'lr': 0.0015930522616241021, 'weight_decay': 0.0013035123791853842, 'epochs': 300, 'm': 4.909729556485983}. Best is trial 1 with value: 0.4741225222555683.
[I 2026-10-19 01:51:00,319] Trial 3 finished with value: 0.4390285541477186 and parameters: {'lr': 0.004622589001020831, 'weight_decay': 0.0001329377199163636, 'epochs': 350, 'm': 2.5502135295603017}. Best is trial 1 with value: 0.4741225222555683.
[I 

GAT text-only                    CV PR-AUC=0.474  {'lr': 0.0002051338263087451, 'weight_decay': 0.00010255552094216992, 'epochs': 300, 'm': 4.598528437324806}


[I 2026-10-19 02:04:34,162] Trial 0 finished with value: 0.43288125835275304 and parameters: {'lr': 0.0005611516415334506, 'weight_decay': 0.0039847274093219665, 'epochs': 550, 'm': 3.79597545259111}. Best is trial 0 with value: 0.43288125835275304.
[I 2026-10-19 02:04:55,320] Trial 1 finished with value: 0.4734042730618744 and parameters: {'lr': 0.0002051338263087451, 'weight_decay': 0.00010255552094216992, 'epochs': 300, 'm': 4.598528437324806}. Best is trial 1 with value: 0.4734042730618744.
[I 2026-10-19 02:05:16,362] Trial 2 finished with value: 0.4355669944493321 and parameters: {'lr': 0.0015930522616241021, 'weight_decay': 0.0013035123791853842, 'epochs': 300, 'm': 4.909729556485983}. Best is trial 1 with value: 0.4734042730618744.
[I 2026-10-19 02:05:41,079] Trial 3 finished with value: 0.44567054591909455 and parameters: {'lr': 0.004622589001020831, 'weight_decay': 0.0001329377199163636, 'epochs': 350, 'm': 2.5502135295603017}. Best is trial 1 with value: 0.4734042730618744.
[

...

[I 2026-10-19 02:35:38,399] Trial 0 finished with value: 0.49350702671109664 and parameters: {'lr': 0.0005611516415334506, 'weight_decay': 0.0039847274093219665, 'epochs': 550, 'm': 3.79597545259111}. Best is trial 0 with value: 0.49350702671109664.
[I 2026-10-19 02:36:01,911] Trial 1 finished with value: 0.47538341873708206 and parameters: {'lr': 0.0002051338263087451, 'weight_decay': 0.00010255552094216992, 'epochs': 300, 'm': 4.598528437324806}. Best is trial 0 with value: 0.49350702671109664.
[I 2026-10-19 02:36:25,254] Trial 2 finished with value: 0.482672594224869 and parameters: {'lr': 0.0015930522616241021, 'weight_decay': 0.0013035123791853842, 'epochs': 300, 'm': 4.909729556485983}. Best is trial 0 with value: 0.49350702671109664.
[I 2026-10-19 02:36:52,634] Trial 3 finished with value: 0.5242167205986143 and parameters: {'lr': 0.004622589001020831, 'weight_decay': 0.0001329377199163636, 'epochs': 350, 'm': 2.5502135295603017}. Best is trial 3 with value: 0.5242167205986143.


GAT-GCN-FeExtr (Empath graph)    CV PR-AUC=0.558  {'lr': 0.0030407178709186385, 'weight_decay': 8.810302346794737e-05, 'epochs': 600, 'm': 2.677641849102402}


[I 2026-10-19 02:55:37,220] Trial 0 finished with value: 0.4400357491536462 and parameters: {'lr': 0.0005611516415334506, 'weight_decay': 0.0039847274093219665, 'epochs': 550, 'm': 3.79597545259111}. Best is trial 0 with value: 0.4400357491536462.
[I 2026-10-19 02:56:01,012] Trial 1 finished with value: 0.4861386017361091 and parameters: {'lr': 0.0002051338263087451, 'weight_decay': 0.00010255552094216992, 'epochs': 300, 'm': 4.598528437324806}. Best is trial 1 with value: 0.4861386017361091.
[I 2026-10-19 02:56:24,922] Trial 2 finished with value: 0.4423130268072424 and parameters: {'lr': 0.0015930522616241021, 'weight_decay': 0.0013035123791853842, 'epochs': 300, 'm': 4.909729556485983}. Best is trial 1 with value: 0.4861386017361091.
[I 2026-10-19 02:56:52,513] Trial 3 finished with value: 0.46392612410228573 and parameters: {'lr': 0.004622589001020831, 'weight_decay': 0.0001329377199163636, 'epochs': 350, 'm': 2.5502135295603017}. Best is trial 1 with value: 0.4861386017361091.
[I 

GAT-GCN-FeExtr (word graph)      CV PR-AUC=0.491  {'lr': 0.0001974679003863818, 'weight_decay': 0.00022601745015481687, 'epochs': 350, 'm': 4.4189410240621845}


1603

## Cross-fitted ensembles, scored once on each held-out cohort

In [ ]:
import numpy as np, pandas as pd
from src.paper1_graph_model.experiment import run_final

def row(model, cohort, m):
    lo, hi = m['f1_macro_ci']
    return {'model': model, 'cohort': cohort, 'n': m['n'], 'positives': m['positives'],
            'tau': round(m['threshold'], 2), 'F1 macro': round(m['f1_macro'], 4),
            'F1 95% CI': f'[{lo:.3f}, {hi:.3f}]', 'recall': round(m['recall'], 3),
            'precision': round(m['precision'], 3), 'specificity': round(m['specificity'], 3),
            'ROC-AUC': round(m.get('roc_auc', np.nan), 3), 'PR-AUC': round(m.get('pr_auc', np.nan), 3),
            'FN': m['fn'], 'FP': m['fp']}

def rows_of(name, res):
    out = [row(name, 'pool (out-of-fold)', res['oof_metrics'])]
    return out + [row(name, c, m) for c, m in res['metrics'].items()]

results, rows = {}, []
for spec in SPECS:
    hp = hp_from_params(spec, best[spec.name]['params'])
    res = run_final(spec, hp, pool, held_out, N_FOLDS, SEEDS_PER_FOLD, SEED, MAX_FPR, DEVICE)
    results[spec.name] = res
    rows += rows_of(spec.name, res)
    print(spec.name, 'tau =', res['tau'], res['tau_info'])
metrics = pd.DataFrame(rows)
metrics.to_csv(RESULTS / 'metrics.csv', index=False)
metrics

InductGCN tau = 0.6 {'fn': 18, 'fp': 30, 'feasible': False, 'max_fpr': 0.2857142857142857}
GAT text-only tau = 0.6 {'fn': 18, 'fp': 30, 'feasible': False, 'max_fpr': 0.2857142857142857}
GCN+GAT hybrid tau = 0.6 {'fn': 18, 'fp': 28, 'feasible': True, 'max_fpr': 0.2857142857142857}
EmpathGCN tau = 0.47 {'fn': 18, 'fp': 24, 'feasible': True, 'max_fpr': 0.2857142857142857}
GAT-GCN-FeExtr (Empath graph) tau = 0.46 {'fn': 19, 'fp': 27, 'feasible': True, 'max_fpr': 0.2857142857142857}
GAT-GCN-FeExtr (word graph) tau = 0.6 {'fn': 18, 'fp': 29, 'feasible': False, 'max_fpr': 0.2857142857142857}


,model,cohort,n,positives,tau,F1 macro,F1 95% CI,recall,precision,specificity,ROC-AUC,PR-AUC,FN,FP
0,InductGCN,pool (out-of-fold),140,42,0.60,0.6196,"[0.533, 0.698]",0.571,0.444,0.694,0.676,0.532,18,30
1,InductGCN,daic_test,46,14,0.60,0.6734,"[0.518, 0.802]",0.714,0.500,0.688,0.650,0.482,4,10
2,InductGCN,edaic,112,34,0.60,0.7130,"[0.618, 0.797]",0.706,0.558,0.756,0.758,0.652,10,19
3,GAT text-only,pool (out-of-fold),140,42,0.60,0.6196,"[0.536, 0.695]",0.571,0.444,0.694,0.681,0.538,18,30
4,GAT text-only,daic_test,46,14,0.60,0.5984,"[0.450, 0.732]",0.571,0.421,0.656,0.636,0.460,6,11
5,GAT text-only,edaic,112,34,0.60,0.6814,"[0.591, 0.764]",0.647,0.524,0.744,0.743,0.646,12,20
6,GCN+GAT hybrid,pool (out-of-fold),140,42,0.60,0.6317,"[0.547, 0.708]",0.571,0.462,0.714,0.679,0.546,18,28
7,GCN+GAT hybrid,daic_test,46,14,0.60,0.6541,"[0.505, 0.781]",0.714,0.476,0.656,0.663,0.479,4,11
8,GCN+GAT hybrid,edaic,112,34,0.60,0.6853,"[0.591, 0.771]",0.676,0.523,0.731,0.773,0.643,11,21
9,EmpathGCN,pool (out-of-fold),140,42,0.47,0.6561,"[0.568, 0.735]",0.571,0.500,0.755,0.680,0.539,18,24


In [ ]:
# Per-participant probabilities stay outside the repository.
for name, res in results.items():
    for c in held_out:
        out = pd.DataFrame({'Participant_ID': c.ids, 'label': c.y, 'prob': res['probs'][c.name]})
        out.to_csv(f"{OUTPUT_DIR}/probs_{name.replace(' ', '_')}_{c.name}.csv", index=False)

## Ablation of the professional-feature views (main model, hyperparameters fixed)

In [ ]:
from dataclasses import replace
MAIN = 'GAT-GCN-FeExtr (Empath graph)'   # change after looking at the table above if needed
main = next(s for s in SPECS if s.name == MAIN)
hp_main = hp_from_params(main, best[main.name]['params'])
ablation = []
for drop in ALL + (None,):
    views = tuple(v for v in ALL if v != drop)
    spec = replace(main, name=f'without {drop}' if drop else 'all views', views=views)
    ablation += rows_of(spec.name, run_final(spec, hp_main, pool, held_out, N_FOLDS, SEEDS_PER_FOLD, SEED, MAX_FPR, DEVICE))
ablation = pd.DataFrame(ablation)
ablation.to_csv(RESULTS / 'ablation_views.csv', index=False)
ablation

,model,cohort,n,positives,tau,F1 macro,F1 95% CI,recall,precision,specificity,ROC-AUC,PR-AUC,FN,FP
0,without temporal,pool (out-of-fold),140,42,0.50,0.6011,"[0.514, 0.683]",0.500,0.429,0.714,0.664,0.512,21,28
1,without temporal,daic_test,46,14,0.50,0.6862,"[0.544, 0.808]",0.857,0.500,0.625,0.688,0.397,2,12
2,without temporal,edaic,112,34,0.50,0.7169,"[0.628, 0.800]",0.824,0.538,0.692,0.807,0.554,6,24
3,without paralinguistic,pool (out-of-fold),140,42,0.50,0.6663,"[0.583, 0.748]",0.595,0.510,0.755,0.679,0.526,17,24
4,without paralinguistic,daic_test,46,14,0.50,0.7310,"[0.596, 0.848]",0.929,0.542,0.656,0.781,0.512,1,11
5,without paralinguistic,edaic,112,34,0.50,0.7125,"[0.613, 0.799]",0.588,0.606,0.833,0.807,0.614,14,13
6,without psycholinguistic,pool (out-of-fold),140,42,0.47,0.5906,"[0.502, 0.672]",0.476,0.417,0.714,0.634,0.552,22,28
7,without psycholinguistic,daic_test,46,14,0.47,0.7718,"[0.630, 0.882]",0.929,0.591,0.719,0.839,0.610,1,9
8,without psycholinguistic,edaic,112,34,0.47,0.6883,"[0.594, 0.783]",0.588,0.556,0.795,0.798,0.627,14,16
9,all views,pool (out-of-fold),140,42,0.43,0.6216,"[0.531, 0.707]",0.548,0.451,0.714,0.688,0.547,19,28


## DSM-5 mapping vs. legacy Empath domain groups (EmpathGCN and main model)

Rebuilds the features with the legacy `EMPATH_DOMAIN_GROUPS` and repeats the final runs with the same hyperparameters.

In [ ]:
cohorts_dom = build_cohorts(data.data_root(), scorer, psycholinguistic='domains')
pool_dom = merge_cohorts('daic_pool', cohorts_dom['daic_train'], cohorts_dom['daic_dev'])
held_dom = [cohorts_dom['daic_test'], cohorts_dom['edaic']]
cmp_rows = []
for name in ['EmpathGCN', MAIN]:
    spec = next(s for s in SPECS if s.name == name)
    hp = hp_from_params(spec, best[name]['params'])
    res_dom = run_final(spec, hp, pool_dom, held_dom, N_FOLDS, SEEDS_PER_FOLD, SEED, MAX_FPR, DEVICE)
    cmp_rows += rows_of(f'{name} / DSM-5 mapping', results[name]) + rows_of(f'{name} / legacy domains', res_dom)
mapping_vs_domains = pd.DataFrame(cmp_rows)
mapping_vs_domains.to_csv(RESULTS / 'mapping_vs_domains.csv', index=False)
mapping_vs_domains

,model,cohort,n,positives,tau,F1 macro,F1 95% CI,recall,precision,specificity,ROC-AUC,PR-AUC,FN,FP
0,EmpathGCN / DSM-5 mapping,pool (out-of-fold),140,42,0.47,0.6561,"[0.568, 0.735]",0.571,0.500,0.755,0.680,0.539,18,24
1,EmpathGCN / DSM-5 mapping,daic_test,46,14,0.47,0.5965,"[0.440, 0.732]",0.714,0.417,0.562,0.688,0.455,4,14
2,EmpathGCN / DSM-5 mapping,edaic,112,34,0.47,0.7083,"[0.608, 0.792]",0.735,0.543,0.731,0.780,0.580,9,21
3,EmpathGCN / legacy domains,pool (out-of-fold),140,42,0.45,0.6415,"[0.552, 0.725]",0.595,0.472,0.714,0.679,0.533,17,28
4,EmpathGCN / legacy domains,daic_test,46,14,0.45,0.5965,"[0.440, 0.732]",0.714,0.417,0.562,0.683,0.456,4,14
5,EmpathGCN / legacy domains,edaic,112,34,0.45,0.6762,"[0.581, 0.761]",0.735,0.500,0.679,0.768,0.566,9,25
6,GAT-GCN-FeExtr (Empath graph) / DSM-5 mapping,pool (out-of-fold),140,42,0.46,0.6277,"[0.541, 0.717]",0.548,0.460,0.724,0.689,0.548,19,27
7,GAT-GCN-FeExtr (Empath graph) / DSM-5 mapping,daic_test,46,14,0.46,0.7262,"[0.589, 0.846]",0.857,0.545,0.688,0.792,0.536,2,10
8,GAT-GCN-FeExtr (Empath graph) / DSM-5 mapping,edaic,112,34,0.46,0.6937,"[0.595, 0.786]",0.529,0.600,0.846,0.810,0.613,16,12
9,GAT-GCN-FeExtr (Empath graph) / legacy domains,pool (out-of-fold),140,42,0.47,0.6415,"[0.552, 0.720]",0.595,0.472,0.714,0.674,0.540,17,28


## Attention over Empath categories (auditability)

Mean attention from each category node to each test participant, averaged over heads and the 20 members.
Per-participant matrices go to `OUTPUT_DIR`; only the class-level ranking is written to the repository.

In [ ]:
from src.paper1_graph_model.experiment import attention_to_vocabulary
test_c = cohorts['daic_test']
att = attention_to_vocabulary(results[main.name], main, test_c, DEVICE)
att.to_csv(f"{OUTPUT_DIR}/attention_test_{main.name.replace(' ', '_')}.csv")
by_class = pd.DataFrame({'control': att[test_c.y == 0].mean(), 'depressed': att[test_c.y == 1].mean()})
by_class['delta'] = by_class['depressed'] - by_class['control']
by_class = by_class.sort_values('delta', ascending=False)
by_class.to_csv(RESULTS / 'attention_by_class_test.csv')
by_class.head(20)

,control,depressed,delta
neglect,0.005902,0.011320,0.005418
health,0.007744,0.011644,0.003899
morning,0.006901,0.010389,0.003487
furniture,0.002458,0.005941,0.003482
aggression,0.007255,0.010267,0.003011
play,0.016272,0.019196,0.002924
gain,0.004198,0.007055,0.002857
timidity,0.009263,0.012087,0.002824
real_estate,0.007126,0.009704,0.002578
occupation,0.009473,0.012028,0.002555
